# Two-term Boltzmann solver

Fluid and global models need the electron transport coefficients and rate
coefficients as functions of the reduced field $E/N$ or of the mean electron
energy. `solve_eedf` computes them from the same `CollisionProcess` objects the
Monte Carlo collisions use, so that every model of a discharge works with rates
consistent with one cross-section set.

It solves the classical two-term approximation of the Boltzmann equation for
electrons in a uniform DC field, following Hagelaar and Pitchford, Plasma
Sources Sci. Technol. 14, 722 (2005), the method of BOLSIG+. The energy
distribution $F_0(\varepsilon)$, normalised by
$\int \varepsilon^{1/2} F_0\, d\varepsilon = 1$, obeys

$$\frac{d}{d\varepsilon}\left(\tilde W F_0 - \tilde D \frac{dF_0}{d\varepsilon}\right) = \tilde S,
\qquad \tilde W = -\gamma\varepsilon^2\sigma_\varepsilon,
\qquad \tilde D = \frac{\gamma}{3}\left(\frac{E}{N}\right)^2\frac{\varepsilon}{\tilde\sigma_m} + \frac{\gamma k T}{e}\,\varepsilon^2\sigma_\varepsilon,$$

with $\gamma = (2e/m_e)^{1/2}$, $\sigma_\varepsilon = \sum_k x_k (2m_e/M_k)\sigma_k$ the
elastic energy-loss cross section, $\tilde\sigma_m$ the total momentum-transfer
cross section (plus the growth term) and $\tilde S$ the inelastic collisions:
excitation moves electrons down by the energy loss, ionization turns one
electron into two, attachment removes one.

In [ ]:
import plasma_plots as pp

pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

## A model gas

The cross sections below are a **synthetic argon-like model gas**, not measured
data: an elastic momentum-transfer cross section, one excitation at 11.5 eV and
an ionization at 15.8 eV. They are written in LXCat format and read back, the
path a real data set from [LXCat](https://lxcat.net) takes.

In [ ]:
from pathlib import Path

import numpy as np

import plasmacoll
from plasmacoll.constants import ATOMIC_MASS, ELECTRON_MASS, kelvin_to_ev

energy = np.round(np.geomspace(1e-2, 1e3, 120), 5)


def block(kind, target, parameter, sigma, process):
    lines = [kind, target, f" {parameter:.6e}", f"PROCESS: {process}", "-----"]
    lines += [f" {e:.6e}\t{s:.6e}" for e, s in zip(energy, sigma, strict=True)]
    return "\n".join([*lines, "-----"])


M = 40 * ATOMIC_MASS
x = np.clip(energy / 15.8, 1, None)
Path("model_gas.txt").write_text(
    "\n\n".join(
        [
            "SYNTHETIC MODEL GAS: these numbers are not measured data.",
            block(
                "ELASTIC",
                "Gas",
                ELECTRON_MASS / M,
                8e-20 / (1 + (energy / 10) ** 0.8),
                "E + Gas -> E + Gas, Elastic",
            ),
            block(
                "EXCITATION",
                "Gas -> Gas*",
                11.5,
                np.where(
                    energy > 11.5,
                    2e-21 * (energy - 11.5) / (1 + (energy / 30) ** 1.5),
                    0,
                ),
                "E + Gas -> E + Gas*, Excitation",
            ),
            block(
                "IONIZATION",
                "Gas -> Gas^+",
                15.8,
                np.where(energy > 15.8, 3e-20 * np.log(x) / x, 0),
                "E + Gas -> E + E + Gas+, Ionization",
            ),
        ]
    )
)
table = plasmacoll.read_lxcat("model_gas.txt")
Path("model_gas.txt").unlink()

gas = plasmacoll.NeutralBackground("gas", 1e22, kelvin_to_ev(300.0), M)
processes = [
    plasmacoll.CollisionProcess(
        "elastic", "gas", table[0].cross_section, name="elastic"
    ),
    plasmacoll.CollisionProcess(
        "excitation", "gas", table[1].cross_section, name="excitation"
    ),
    plasmacoll.CollisionProcess(
        "ionization",
        "gas",
        table[2].cross_section,
        name="ionization",
        products={"electron": "e", "ion": "ion"},
    ),
]

## The energy distribution at a few fields

`solve_eedf(processes, mixture, E/N)` takes $E/N$ in Td
($1\,\mathrm{Td} = 10^{-21}\,\mathrm{V\,m^2}$) and the gas temperature from the
mixture. It chooses the energy grid so that the distribution falls by 10 to 20
decades across it. Plotted is the energy probability function
$F_0 = f(\varepsilon)/\varepsilon^{1/2}$, on which a Maxwellian is a straight line:
above the excitation threshold the inelastic collisions deplete the tail.

In [ ]:
with pp.figure(
    title="Inelastic collisions bend the distribution down above 11.5 eV"
) as fig:
    ax = fig[0]
    for reduced_field in (10, 30, 100, 300):
        solution = plasmacoll.solve_eedf(processes, [gas], reduced_field)
        ax.semilogy(
            solution.energy,
            solution.eepf,
            label=f"{reduced_field} Td, $\\langle \\varepsilon \\rangle$ = {solution.mean_energy:.1f} eV",
        )
    ax.axvline(11.5, color="0.4", linewidth=1, linestyle=":")
    ax.set_xlim(0, 60)
    ax.set_ylim(1e-7, 1)
    ax.set_xlabel(r"electron energy $\varepsilon$ [eV]")
    ax.set_ylabel("EEPF $F_0$ [eV$^{-3/2}$]")
    ax.legend(loc="upper right")

Besides the distribution, a solution has the mean energy, the reduced mobility
$\mu N$ and diffusion coefficient $DN$, the rate coefficient of every process
and the reduced Townsend coefficient $\alpha/N$. Its power balance compares the
power gained from the field with the power lost in collisions, and is a measure
of the discretisation error:

In [ ]:
solution = plasmacoll.solve_eedf(processes, [gas], 100.0)
print(f"mean energy      {solution.mean_energy:.3f} eV")
print(f"mu N             {solution.mobility:.4g} 1/(V m s)")
print(f"D N              {solution.diffusion:.4g} 1/(m s)")
print(f"drift velocity   {solution.drift_velocity:.4g} m/s")
for name, rate in solution.rate_coefficients.items():
    print(f"k {name:14s} {rate:.4g} m^3/s")
print(f"alpha/N          {solution.townsend_coefficient:.4g} m^2")
print(f"power balance    {solution.power_balance:.1e}")

## Rates against the mean energy

`swarm_table` solves over a range of fields and tabulates the results. Fluid
models look the coefficients up against the mean energy, which
`SwarmTable.interpolate` does, and `SwarmTable.rate_table` returns a rate as a
`RateTable` against the effective temperature $\tfrac23\langle\varepsilon\rangle$,
the form global models use.

In [ ]:
swarm = plasmacoll.swarm_table(processes, [gas], np.geomspace(0.3, 1000, 25))

with pp.figure(title="Rate coefficients") as fig:
    ax = fig[0]
    for name in ("elastic", "excitation", "ionization"):
        rates = swarm.rate_coefficients[name]
        shown = rates > 1e-22
        ax.loglog(swarm.mean_energy[shown], rates[shown], label=name)
    ax.set(
        xlabel=r"mean energy $\langle \varepsilon \rangle$ [eV]",
        ylabel="rate coefficient $k$ [m$^3$/s]",
    )
    ax.legend()
with pp.figure(title="Transport") as fig:
    ax = fig[0]
    ax.loglog(swarm.mean_energy, swarm.mobility, label=r"$\mu N$ [1/(V m s)]")
    ax.loglog(swarm.mean_energy, swarm.diffusion, label=r"$D N$ [1/(m s)]")
    ax.set(
        xlabel=r"mean energy $\langle \varepsilon \rangle$ [eV]",
        ylabel=r"reduced mobility $\mu N$ and diffusion $D N$",
    )
    ax.legend()

ionization = swarm.rate_table("ionization")
print(f"k_iz at <ε> = 6 eV:         {swarm.interpolate('ionization', 6.0):.3g} m^3/s")
print(f"RateTable at Te = 4 eV:     {ionization(4.0):.3g} m^3/s")
print(f"E/N for <ε> = 6 eV:         {swarm.interpolate('reduced_field', 6.0):.1f} Td")

## Levels and superelastic collisions

In a molecular gas the electrons also collide with excited molecules and gain
their energy back. `solve_eedf` takes the levels of a gas as `states`, each a
fraction of the gas, and an excitation with an `upper_state` gets its inverse,
the superelastic collision, with the cross section of detailed balance

$$\sigma^\mathrm{sup}(\varepsilon) = \frac{g_l}{g_u}\left(1 + \frac{\Delta E}{\varepsilon}\right)\sigma(\varepsilon + \Delta E).$$

With the levels populated at the gas temperature this is how the gas temperature
enters the inelastic collisions. The gas below is a **synthetic H2-like model**:
rotational levels $J = 0$–9 with H2's rotational constant and nuclear-spin
weights at 300 K, Gerjuoy–Stein $J \to J + 2$ cross sections, a vibrational
excitation at 0.5 eV with $v = 1$ populated at 2000 K, an electronic excitation
and ionization. Processes on the gas itself (elastic, electronic, ionization) act
on all molecules, those on a level on its share.

In [ ]:
from plasmacoll import CollisionProcess, CrossSection

B = 2.0**-7  # eV, the rotational constant (H2: 7.5e-3 eV)
kT, kTv = kelvin_to_ev(300.0), kelvin_to_ev(2000.0)
levels = range(10)


def g(J):  # 2J + 1 times H2's nuclear-spin weight, 1 (para) or 3 (ortho)
    return (2 * J + 1) * (3 if J % 2 else 1)


boltzmann = np.array([g(J) * np.exp(-B * J * (J + 1) / kT) for J in levels])
v1 = np.exp(-0.5 / kTv) / (1 + np.exp(-0.5 / kTv))
states = {"v=0": ("H2", 1 - v1), "v=1": ("H2", v1)}
states |= {
    f"J={J}": ("H2", (1 - v1) * p)
    for J, p in zip(levels, boltzmann / boltzmann.sum(), strict=True)
}


def above(threshold, sigma):
    e = threshold + np.concatenate([[0.0], np.geomspace(1e-5, 1e3 - threshold, 200)])
    return CrossSection(e, np.where(e > threshold, sigma(e), 0.0), threshold=threshold)


def bethe(threshold, peak):
    return lambda e: peak * np.e * np.log(e / threshold) / (e / threshold)


e = np.concatenate([[0.0], np.geomspace(1e-3, 1e3, 300)])
elastic = 6e-20 * (1 + 2 * e / (1 + e)) / (1 + (e / 15) ** 1.2)
quadrupole = 8 * np.pi * 0.47**2 * 5.2917721067e-11**2 / 15  # Gerjuoy-Stein, m^2
h2 = [
    CollisionProcess("elastic", "H2", CrossSection(e, elastic), name="elastic"),
    CollisionProcess(
        "excitation",
        "v=0",
        above(0.5, lambda e: 5e-21 * (e - 0.5) / 2.5 * np.exp(1 - (e - 0.5) / 2.5)),
        name="v=0->1",
        upper_state="v=1",
    ),
    CollisionProcess("excitation", "H2", above(11.25, bethe(11.25, 4e-21)), name="B"),
    CollisionProcess(
        "ionization",
        "H2",
        above(15.5, bethe(15.5, 1e-20)),
        name="ionization",
        products={"electron": "e", "ion": "H2+"},
    ),
]
for J in range(8):
    loss = B * (4 * J + 6)
    strength = quadrupole * (J + 2) * (J + 1) / ((2 * J + 3) * (2 * J + 1))
    h2.append(
        CollisionProcess(
            "excitation",
            f"J={J}",
            above(loss, lambda e, u=loss, s=strength: s * np.sqrt(1 - u / e)),
            name=f"J={J}->{J + 2}",
            upper_state=f"J={J + 2}",
            statistical_weight_ratio=g(J + 2) / g(J),
        )
    )
hydrogen = plasmacoll.NeutralBackground("H2", 1e22, kT, 2.016 * ATOMIC_MASS)

[LoKI-B](https://github.com/LoKI-Suite/LoKI-B-cpp), the LisbOn KInetics Boltzmann
solver (Tejero-del-Caz et al., Plasma Sources Sci. Technol. 28, 043001 (2019)),
solved the same gas; its results come with plasmacoll's tests. The lines are
`solve_eedf`, the circles LoKI-B, and the dashed line the 1 Td distribution
without superelastic collisions: the hot vibrations and the rotations at 300 K
heat the electrons at low fields.

In [ ]:
import dataclasses
import json

from plasmacoll.tests.unit import _lokib_models

reference = json.loads(_lokib_models.REFERENCE.read_text())["cases"]
print("E/N (Td)   <ε> (eV) plasmacoll / LoKI-B   μN (1/(V m s)) plasmacoll / LoKI-B")
solutions = {}
for reduced_field in (1, 10, 30, 100):
    s = plasmacoll.solve_eedf(h2, [hydrogen], reduced_field, states=states)
    lokib = reference[f"mol_{reduced_field}"]
    solutions[reduced_field] = s
    print(
        f"{reduced_field:8d}   {s.mean_energy:.4f} / {lokib['mean_energy']:.4f}"
        f"            {s.mobility:.5g} / {lokib['mobility']:.5g}"
    )
cold = [dataclasses.replace(p, upper_state=None) for p in h2]
no_superelastic = plasmacoll.solve_eedf(cold, [hydrogen], 1, states=states)

for title, fields, xmax in (
    ("Low fields: superelastic heating", (1, 10), 5),
    ("High fields", (30, 100), 40),
):
    with pp.figure(title=f"{title} (lines: solve_eedf, circles: LoKI-B)") as fig:
        ax = fig[0]
        for index, reduced_field in enumerate(fields):
            s, lokib = solutions[reduced_field], reference[f"mol_{reduced_field}"]
            color = f"C{index}"
            ax.semilogy(s.energy, s.eepf, color=color, label=f"{reduced_field} Td")
            ax.semilogy(
                lokib["eedf_energy"], lokib["eedf"], "o", color=color, mfc="none"
            )
        if 1 in fields:
            ax.semilogy(
                no_superelastic.energy,
                no_superelastic.eepf,
                "--",
                color="C0",
                label="1 Td, no superelastic",
            )
        ax.set(xlim=(0, xmax), ylim=(1e-9, 10))
        ax.set_xlabel(r"electron energy $\varepsilon$ [eV]")
        ax.set_ylabel("EEPF $F_0$ [eV$^{-3/2}$]")
        ax.legend(loc="upper right")

## Checks against published results

Reid's ramp model gas (Aust. J. Phys. 32, 231 (1979)) is the standard
benchmark of Boltzmann solvers: a constant elastic cross section of
$6\,\text{Å}^2$, an excitation of $10(\varepsilon - 0.2)\,\text{Å}^2/\mathrm{eV}$
above 0.2 eV, $M = 4$ u and a cold gas. The reference values are the two-term
results of Flynn et al., J. Phys. D 57, 255204 (2024), Table 1.

In [ ]:
reid_gas = plasmacoll.NeutralBackground("gas", 1e22, 0.0, 4 * ATOMIC_MASS)
reid = [
    plasmacoll.CollisionProcess(
        "elastic", "gas", plasmacoll.CrossSection.constant(6e-20)
    ),
    plasmacoll.CollisionProcess(
        "excitation",
        "gas",
        plasmacoll.CrossSection([0.2, 1000.2], [0.0, 1e-16], threshold=0.2),
    ),
]
published = {  # E/N: mean energy (eV), drift velocity (m/s), D N (1/(m s))
    1: (0.1015, 1.2750e4, 0.9905e24),
    12: (0.2735, 7.0290e4, 1.3690e24),
    24: (0.4162, 9.1447e4, 1.4382e24),
}
print("E/N   <ε> (eV)          w (m/s)              D N (1/(m s))")
for reduced_field, (mean, drift, diffusion) in published.items():
    s = plasmacoll.solve_eedf(reid, [reid_gas], reduced_field)
    print(
        f"{reduced_field:3d}   {s.mean_energy:.4f} / {mean:.4f}   "
        f"{s.drift_velocity:.4g} / {drift:.4g}   {s.diffusion:.4g} / {diffusion:.4g}"
    )

The test suite also checks the Maxwell and Druyvesteyn distributions in their
limits, the Lucas–Saelee ionization model gas, that the discretisation loses no
electrons and closes the energy balance to second order in the cell size,
agreement with LoKI-B on the H2-like gas above and on an argon-like gas with
both growth models and both ways of sharing the energy in an ionization (within
LoKI-B's own discretisation error, 0.01 to 0.1 %), and, where
[BOLOS](https://github.com/aluque/bolos) is installed, agreement with BOLOS on
the model gas above to better than 0.5 %.

The two-term approximation itself fails where the distribution is strongly
anisotropic, when inelastic collisions dominate the momentum transfer; multi-term
results for the Reid ramp differ by a few per cent in the drift velocity and by
about 20 % in the diffusion coefficient. For a plasma model's rates it is
usually accurate enough.